# Circos Plot Visualization of GWAS Association Data
**Tool:** PCircos (Python wrapper for Circos-style circular genome plots, built on Plotly)  
**Data:** GWAS association results, formatted into scatter and heatmap tracks

---

## Overview

Circos plots are a classic way to visualize genome-wide data along a circular ideogram, where each segment of the circle represents a chromosome. They're especially useful for showing multiple layers of genomic information simultaneously — for example, association strength, variant density, and structural features — all on one figure.

This notebook uses **PCircos**, a Python/Plotly-based reimplementation of the original Circos software (which was written in Perl and required significant configuration overhead). PCircos lets you generate publication-style circular plots from simple text/CSV input files and a JSON configuration file, with the added benefit of interactive hover tooltips since it's built on Plotly.

**What we'll build:**
1. A scatter track showing GWAS p-values positioned around the genome by chromosome
2. A heatmap track showing variant density across genomic windows
3. A combined multi-track Circos plot

GitHub repository: https://github.com/CJinnny/PCircos

## 1. Setup

In [ ]:
# Clone the PCircos repository
!rm -rf PCircos
!git clone -q https://github.com/CJinnny/PCircos.git

In [ ]:
# Install PCircos's dependencies
# Note: these are pinned to older versions required by PCircos's plotting backend
!pip install plotly==3.6.1 -q
!pip install colorlover -q
!pip install dash==0.39.0 -q
!pip install dash-daq==0.1.0 -q
!pip install dash_colorscales -q

In [ ]:
import os
import pandas as pd
from IPython.display import HTML

os.chdir('/content/PCircos')

## 2. Running the Default Demo

Before working with our own data, let's confirm PCircos runs correctly using its bundled demo configuration. `demo_params.json` is a configuration file that specifies plot styling, track layout, and data file locations — PCircos reads this file and renders the corresponding figure as an HTML file.

In [ ]:
!python PCircos.py demo_data/demo_params.json
HTML(filename='temp-plot.html')

### 2.1 Understanding the configuration structure

A PCircos `params.json` file has two main sections:

- **General** — controls plot width, height, title, and font settings. If omitted, sensible defaults are used (see the [default config](https://github.com/CJinnny/PCircos/blob/master/config/default_params.json) for reference).
- **Category** — defines each data track: its type (scatter, heatmap, line, etc.), input file path, and visual styling (color, size, opacity).

With the demo confirmed working, we can now build our own tracks from real GWAS data.

## 3. Preparing GWAS Data for Circos Format

PCircos expects each data track in a specific tab-separated format:

```
chr_name    start    end    value
```

Chromosome names must be prefixed with `hs` (e.g., `hs1` for chromosome 1). We'll transform a standard GWAS association file into this format using command-line tools — a common step in genomics pipelines where raw analysis output rarely matches a visualization tool's expected input format directly.

In [ ]:
# Download example GWAS association data
!wget -q 'https://raw.githubusercontent.com/phagenomics/kgi-demo/main/AssocDemo.txt' -O demo.txt

In [ ]:
# Inspect the raw file before transforming it
!wc -l demo.txt
print()
!head -5 demo.txt

### 3.1 Build the scatter track

We extract chromosome, position, and p-value columns, format chromosome names with the required `hs` prefix, and limit to the first 1,000 variants to keep the plot readable. Color is set to red for all points here, but could be conditioned on significance threshold instead.

In [ ]:
# Transform: skip header, take first 1000 variants, sort by chromosome,
# reformat as: hs<CHR>  <BP>  <P>  red
!grep -v 'CHR' demo.txt | head -n 1000 | sort -k 1 -g | \
    awk '{print "hs"$1"\t"$3"\t"$9"\tred"}' > TMP

# Prepend the expected header from PCircos's own scatter track template
!head -n 1 demo_data/scatter.txt > HEAD
!cat HEAD TMP > demo_data/new_scatter.txt

print("Scatter track preview:")
!head -5 demo_data/new_scatter.txt

### 3.2 Build the heatmap track

For the heatmap track, we bin variants into 100kb genomic windows (start to start+100,000), again limited to the first 1,000 variants for this example.

In [ ]:
!grep -v 'CHR' demo.txt | head -n 1000 | sort -k 1 -g | \
    awk '{print "hs"$1"\t"$3"\t"$3+100000"\t1"}' > TMP

!head -n 1 demo_data/heatmap.txt > HEAD
!cat HEAD TMP > demo_data/new_heatmap.txt

print("Heatmap track preview:")
!head -5 demo_data/new_heatmap.txt

### 3.3 Build a second heatmap track from a different variant subset

To demonstrate multi-track comparison, we build a second heatmap track using the *last* 1,000 variants in the file instead of the first — useful for comparing two different subsets of variants (e.g., two chromosomal regions, or two significance tiers) on the same plot.

In [ ]:
!grep -v 'CHR' demo.txt | tail -n 1000 | sort -k 1 -g | \
    awk '{print "hs"$1"\t"$3"\t"$3+100000"\t1"}' > TMP2

!head -n 1 demo_data/heatmap.txt > HEAD
!cat HEAD TMP2 > demo_data/new_heatmap2.txt

print("Second heatmap track preview:")
!head -5 demo_data/new_heatmap2.txt

## 4. Building a Custom Configuration File

With our data tracks prepared, we write a custom `params.json` that points PCircos at our new files instead of the bundled demo data, combining the scatter track and both heatmap tracks into a single multi-layer plot.

In [ ]:
import json

custom_params = {
    "General": {
        "width": 800,
        "height": 800,
        "title": "GWAS Association Data — Circos Plot"
    },
    "Category": [
        {
            "type": "SCATTER",
            "file": "demo_data/new_scatter.txt",
            "radius": [0.75, 0.95]
        },
        {
            "type": "HEATMAP",
            "file": "demo_data/new_heatmap.txt",
            "radius": [0.55, 0.70]
        },
        {
            "type": "HEATMAP",
            "file": "demo_data/new_heatmap2.txt",
            "radius": [0.35, 0.50]
        }
    ]
}

with open('demo_data/custom_params.json', 'w') as f:
    json.dump(custom_params, f, indent=2)

print("Custom configuration written to demo_data/custom_params.json")
print(json.dumps(custom_params, indent=2))

**Note:** the exact schema for `Category` entries (radius ranges, required fields per track type) is defined by PCircos's own parsing logic — check `config/default_params.json` in the cloned repo if a track doesn't render as expected. Circular plot configuration tends to require some trial and error to get radius ranges that don't overlap awkwardly.

## 5. Rendering the Custom Circos Plot

In [ ]:
!python PCircos.py demo_data/custom_params.json
HTML(filename='temp-plot.html')

The resulting plot shows GWAS p-values as a scatter track around the genome's circular ideogram, with two heatmap tracks beneath it showing variant density for two different subsets of the data. Because PCircos is built on Plotly, hovering over any point or heatmap cell in the live notebook reveals its underlying data values — a meaningful usability improvement over static Circos plots generated by the original Perl tool.

## 6. Summary

This notebook demonstrated an end-to-end workflow for building Circos-style circular genome visualizations from raw GWAS association data:

1. **Setup** — installed PCircos and its plotting dependencies
2. **Validation** — confirmed the tool works using its bundled demo
3. **Data wrangling** — transformed raw GWAS output into PCircos's required tab-separated track format using standard command-line tools (`grep`, `awk`, `sort`, `head`/`tail`)
4. **Configuration** — built a custom JSON config combining a scatter track and two heatmap tracks
5. **Rendering** — generated an interactive, multi-track circular genome plot

**Key takeaways:**
- Visualization tools often require data in a very specific format; a meaningful part of genomics work is data wrangling to bridge that gap, not just running the visualization itself
- Circos-style plots are well suited to displaying multiple genomic data layers (association strength, variant density, etc.) simultaneously in a compact circular layout
- Interactive plotting libraries (Plotly, in this case) add real practical value over static images — being able to hover and inspect individual data points speeds up exploratory analysis

**Potential extensions:**
- Color scatter points by significance threshold rather than uniformly
- Add a links/ribbon track connecting related genomic regions (a common Circos use case for structural variants or chromosomal translocations)
- Apply this same pipeline to a different dataset, such as variant density across a family trio